# InsightStay 유럽 숙박 데이터 EDA**목적** — `InsightStay_data.csv`(약 1GB, 842,655행 × 50열)의 컬럼별 의미·분포·결측·이상치를 전수 조사하고,결측 처리 방안을 **근거와 함께** 정리해 분석용 데이터셋을 확정한다.## 목차| # | 섹션 | 내용 ||---|---|---|| 0 | 환경 설정 & 로딩 | 1GB CSV를 빠르게 읽는 방법, parquet 캐시 || 1 | 데이터 개요 | 크기·중복·타입·고유값 || 2 | 컬럼 사전 | README 기준 50개 컬럼의 의미 || 3 | 결측치 전수 조사 | 결측률 + **결측의 구조(왜 비었나)** || 4 | 키·출처 | `country` / `region` / `source` — source의 정체 규명 || 5 | 호스트 | 경력·슈퍼호스트·포트폴리오 규모 || 6 | 숙소 속성 | room/property type, 수용인원, 침실·욕실, amenities || 7 | 가격 | price / list_price / quote 3종의 관계, 할인, 통화 || 8 | 숙박일 제약 | minimum / maximum_nights, 센티넬 값 || 9 | 예약 가능성 | availability_30/60/90/365 → 가동률 프록시 || 10 | 리뷰 | 리뷰 수·기간·평점 || 11 | 이상치 종합 | 도메인 규칙 기반 플래그 || 12 | **결측치 처리 방안** | 컬럼별 권고안과 논거 || 13 | 분석용 데이터셋 | 권장 필터 + 파생변수 |> 각 코드 셀은 **하나의 질문**만 답하도록 쪼개 두었고, 셀 바로 아래 마크다운에 해석을 적었습니다.

## 0. 환경 설정 & 데이터 로딩

In [ ]:
# [0-1] 라이브러리 & 출력/그래프 기본 설정
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# 한글 폰트 (Windows: 맑은 고딕 / mac: AppleGothic)
for _f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]:
    if any(_f == f.name for f in mpl.font_manager.fontManager.ttflist):
        mpl.rcParams["font.family"] = _f
        break
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["figure.dpi"] = 110
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.alpha"] = 0.25
mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False

C = {"main": "#4E79A7", "accent": "#F28E2B", "ok": "#59A14F", "bad": "#E15759", "gray": "#9C9C9C"}
print("pandas", pd.__version__, "| numpy", np.__version__, "| matplotlib", mpl.__version__)
print("font:", mpl.rcParams["font.family"])

In [ ]:
# [0-2] 데이터 로딩 — 1GB CSV는 pyarrow 엔진으로 읽고 parquet으로 캐시해 재실행을 빠르게 한다.
CSV_PATH = "InsightStay_data.csv"
CACHE    = "_cache_insightstay.parquet"

# name / description 은 순수 자유 텍스트이고 파일 용량의 대부분을 차지한다.
# 수치 EDA에는 불필요하므로 기본 로딩에서 제외한다(텍스트 분석이 필요하면 TEXT_COLS 를 빈 set 으로).
TEXT_COLS = {"name", "description"}

if os.path.exists(CACHE):
    df = pd.read_parquet(CACHE)
    print("parquet 캐시에서 로드")
else:
    all_cols = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
    use_cols = [c for c in all_cols if c not in TEXT_COLS]
    df = pd.read_csv(CSV_PATH, usecols=use_cols, engine="pyarrow")
    df.to_parquet(CACHE, index=False)
    print(f"CSV에서 로드 → {CACHE} 캐시 저장 (제외 컬럼: {sorted(TEXT_COLS)})")

print("shape:", df.shape)
print("메모리:", f"{df.memory_usage(deep=True).sum()/1024**3:,.2f} GB")
df.head(3)

> **로딩 팁** — `engine="pyarrow"` 로 읽으면 1GB CSV가 약 5초에 읽힙니다(기본 C 엔진은 수 분).> `name`, `description` 두 텍스트 컬럼이 파일 용량의 대부분이라 제외하면 메모리가 크게 줄어듭니다.> 이후 모든 셀은 캐시된 parquet 기준으로 동작하므로 노트북 재시작 후에도 즉시 실행됩니다.

## 1. 데이터 개요

In [ ]:
# [1-1] 기본 개요 — 행/열, 키 중복, 호스트 수
n = len(df)
print(f"행(리스팅 수)      : {n:,}")
print(f"열                 : {df.shape[1]} (원본 50열 중 name/description 제외)")
print(f"id 고유값          : {df['id'].nunique():,}  → 중복 {df['id'].duplicated().sum():,}건")
print(f"host_id 고유값     : {df['host_id'].nunique():,}  → 호스트 1인당 평균 {n/df['host_id'].nunique():.2f}개 리스팅")
print(f"국가 / 도시(region): {df['country'].nunique()} / {df['region'].nunique()}")
print(f"완전 중복 행       : {df.duplicated().sum():,}")

In [ ]:
# [1-2] 컬럼별 프로파일 — 타입 / 결측 / 고유값 / 예시값
profile = pd.DataFrame({
    "dtype":       df.dtypes.astype(str),
    "n_missing":   df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(2),
    "nunique":     df.nunique(dropna=True),
})
profile["example"] = [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
profile["example"] = profile["example"].astype(str).str.slice(0, 40)
profile

**한눈에 보는 결론**- `id` 는 완전한 고유키(842,655건, 중복 0). 완전 중복 행도 0건 → 행 단위 중복 제거 불필요.- 호스트 436,124명이 842,655개 리스팅 보유 → **1인당 평균 1.93개**. 뒤에서 보듯 실제로는 소수 전문 사업자가 꼬리를 길게 끕니다.- 결측이 있는 컬럼은 크게 **① 호스트 메타(900건)**, **② 숙소 스펙(bathrooms/bedrooms/beds)**, **③ 가격 5종**, **④ 리뷰 11종**, **⑤ 위치 보조(neighbourhood, host_location)** 5개 덩어리로 묶입니다. 섹션 3에서 각 덩어리의 **발생 원인**을 규명합니다.

## 2. 컬럼 사전 (README 기준 + 실측 의미)| 그룹 | 컬럼 | 의미 | 실측 특징 ||---|---|---|---|| **키·출처** | `country` | 국가 (19개) | 결측 없음 || | `region` | 도시/광역 단위 (53개) | 결측 없음. 런던 92,638 ~ 로테르담 1,357로 규모 편차 68배 || | `id` | 리스팅 고유 ID | 완전 고유. 값 자체가 생성 시점 순서(작을수록 오래된 매물) || | `source` | **데이터 갱신 여부** — 0=새로 수집 / 1=갱신 실패로 과거값 이월 | **분석 설계의 핵심 변수.** 섹션 4 참조 || **호스트** | `host_id` | 호스트 고유 ID | 436,124명 || | `hosts_time_as_user_years/months` | 플랫폼 가입 후 경과 기간 | 연·월이 분리된 형태 → `years + months/12` 로 합쳐 써야 함 || | `hosts_time_as_host_years/months` | **호스팅 시작** 후 경과 기간 | user 경력 ≥ host 경력 (가입 후 호스팅 시작) || | `host_location` | 호스트 거주지(자유 입력) | 27.6% 결측, 13,348개 표기 변형 || | `host_is_superhost` | 슈퍼호스트 여부 `t`/`f` | 문자열. 27.8%가 슈퍼호스트 || **위치·속성** | `neighbourhood_group_cleansed` | 광역 행정구역 | **53개 도시 중 36개가 100% 결측** — 도시별 수집 스키마 차이 || | `property_type` | 건물 유형 (149종) | 상위 3종이 전체의 69% || | `room_type` | 대여 형태 (4종) | Entire home/apt 80.1% || | `accommodates` | 최대 수용 인원 | 1~16, 결측 없음 || | `bathrooms` / `bedrooms` / `beds` | 욕실·침실·침대 수 | 결측 25.1% / 16.2% / 21.7% — 대부분 `source` 탓 || | `amenities` | 편의시설 JSON 배열 문자열 | 결측 없음(단 빈 배열 878건). 중앙값 31개 || **가격** | `price` | **1박 실판매가(할인 반영)** | `price_quote_total_price ÷ 숙박일수` 와 정확히 일치 || | `price_quote_checkin/checkout_date` | 견적 기준 체크인/아웃 날짜 | 2026-06-16 ~ 2027-07-22. 견적 숙박일수(LOS)가 행마다 다름 || | `price_quote_total_price` | 견적 총액 (**청소비·수수료 포함**) | LOS가 1~100박으로 제각각이라 **직접 비교 금지** || | `list_price` | **할인 전 정가** | `price ≤ list_price` || | `discount_type` | 할인 종류 (5종) | 88.9% 결측 = 할인 없음 || **숙박 제약** | `minimum_nights` | 최소 숙박일 | 중앙값 2박, 최대 1,125박 || | `maximum_nights` | 최대 숙박일 | 1125 / 365 에 인위적 집중, 2,147,483,647(=int32 max) 센티넬 존재 || **예약 가능성** | `availability_30/60/90/365` | 향후 N일 중 **예약 가능(빈) 일수** | 값이 작을수록 이미 예약되었거나 호스트가 막아둔 것 || **리뷰** | `number_of_reviews` | 누적 리뷰 수 | 0인 매물 19.7% || | `number_of_reviews_ltm` / `_ly` / `_l30d` | 최근 12개월 / 지난 1년 / 최근 30일 리뷰 수 | ⚠️ `_ltm`과 `_ly`는 **기간이 겹침**(46% 일치). 전년 대비 계산 금지 || | `first_review` / `last_review` | 최초·최종 리뷰일 | 2009-06-20 ~ 2026-07-22 (데이터 기준일 ≈ 2026-07-22) || | `review_scores_rating` + 6개 세부 | 종합 및 정확성·청결·체크인·소통·위치·가성비 평점 | 0~5. 리뷰 0건이면 전부 결측 || | `reviews_per_month` | **최초 리뷰 이후 전체 기간** 기준 월평균 리뷰 수 | ⚠️ 최근 활동량이 아니라 **생애 평균** || **호스트 규모** | `calculated_host_listings_count_entire_homes` / `_private_rooms` / `_shared_rooms` | 같은 호스트의 유형별 매물 수 | 셋을 합치면 호스트 총 매물 수 |

## 3. 결측치 전수 조사

In [ ]:
# [3-1] 결측률 상위 컬럼 — 표 + 막대그래프
miss = (df.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]

fig, ax = plt.subplots(figsize=(8, 0.32 * len(miss) + 1))
colors = [C["bad"] if v >= 50 else C["accent"] if v >= 10 else C["main"] for v in miss.values]
ax.barh(miss.index[::-1], miss.values[::-1], color=colors[::-1])
for y, v in enumerate(miss.values[::-1]):
    ax.text(v + 0.7, y, f"{v:.1f}%", va="center", fontsize=8)
ax.set_xlim(0, 100); ax.set_xlabel("결측률 (%)")
ax.set_title(f"결측이 있는 컬럼 {len(miss)}개 / 전체 {df.shape[1]}개", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

miss.round(2).to_frame("pct_missing")

In [ ]:
# [3-2] 결측의 '구조' 규명 ①  — source(0/1)별 결측률 히트맵
mcols = miss.index.tolist()
by_src = df.groupby("source")[mcols].apply(lambda g: g.isna().mean() * 100).round(1)

fig, ax = plt.subplots(figsize=(0.55 * len(mcols) + 2, 2.8))
ax.imshow(by_src.values, cmap="Reds", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(len(mcols))); ax.set_xticklabels(mcols, rotation=60, ha="right", fontsize=8)
ax.set_yticks([0, 1]); ax.set_yticklabels(["source=0", "source=1"])
for i in range(by_src.shape[0]):
    for j in range(by_src.shape[1]):
        v = by_src.values[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7,
                color="white" if v > 55 else "black")
ax.set_title("source별 결측률(%) — 색이 진할수록 결측", loc="left", fontweight="bold")
ax.grid(False)
plt.tight_layout(); plt.show()

by_src

In [ ]:
# [3-3] 결측의 '구조' 규명 ②  — 결측이 다른 컬럼 값으로 완전히 설명되는지 교차 검증
checks = {
    "price 결측  <->  availability_365 == 0":
        pd.crosstab(df["availability_365"].eq(0), df["price"].isna(),
                    rownames=["avail365==0"], colnames=["price 결측"]),
    "review_scores_rating 결측  <->  number_of_reviews == 0":
        pd.crosstab(df["number_of_reviews"].eq(0), df["review_scores_rating"].isna(),
                    rownames=["리뷰0건"], colnames=["rating 결측"]),
    "bathrooms 결측  <->  source == 1":
        pd.crosstab(df["source"], df["bathrooms"].isna(),
                    rownames=["source"], colnames=["bathrooms 결측"]),
    "discount_type 결측  <->  price == list_price (할인 없음)":
        pd.crosstab(df["discount_type"].isna(), (df["price"] < df["list_price"]),
                    rownames=["discount 결측"], colnames=["price<list"]),
}
for title, t in checks.items():
    print("■", title)
    print(t.to_string(), "\n")

print("※ 호스트 메타(900건) 동시 결측 여부:",
      (df["host_is_superhost"].isna() & df["hosts_time_as_user_years"].isna()).sum(), "/ 900")
print("※ 리뷰 0건인데 평점이 있는 예외:",
      ((df["number_of_reviews"] == 0) & df["review_scores_rating"].notna()).sum(), "건")

### 결측은 "랜덤"이 아니라 **여섯 가지 구조적 원인**에서 나온다| 원인 | 해당 컬럼 | 증거 ||---|---|---|| **(A) 갱신 실패로 과거값 이월(source=1)** | `bathrooms`(100%), `beds`(100%), `price` 계열(79%) | 이월 스냅샷에 필드가 없거나 실시간 견적이라 이월 불가. bathrooms·beds 결측률 **정확히 100%** || **(B) 논리적으로 값이 존재할 수 없음 (구조적 결측)** | `review_scores_*` 7종, `reviews_per_month`, `first/last_review` | 리뷰 0건 165,637건 ↔ 평점 결측 165,630건이 **완전 일치**(예외 7건) || **(C) 예약 불가라서 견적이 산출되지 않음** | `price`, `list_price`, `price_quote_*` | `availability_365 == 0` 인 112,410건 중 **99.9%** 가 price 결측 || **(D) 값 없음 = 의미 있는 상태("해당 없음")** | `discount_type` (88.9%) | 할인 없는 매물은 `price == list_price`. 결측 ↔ 무할인이 100% 대응 || **(E) 도시별 수집 스키마 차이** | `neighbourhood_group_cleansed` (74.7%) | 53개 도시 중 **36개는 100% 결측, 17개는 0% 결측** — 중간이 없음 || **(F) 자유 입력 미기재** | `host_location` (27.6%) | 유일하게 "진짜 랜덤에 가까운" 결측 |**→ 가장 중요한 함의:** `price` 결측은 MAR/MCAR이 아니라 **MNAR(비무작위 결측)** 입니다."가격이 비어 있다 = 예약이 불가능한 매물"이라는 정보 그 자체이므로, **평균/중앙값으로 채우면 안 됩니다.**채우는 순간 "팔리지 않는 매물"이 "평균 가격의 정상 매물"로 둔갑해 수익성 진단이 왜곡됩니다.

## 4. 키·출처 — `country` / `region` / `source`

In [ ]:
# [4-1] 도시(region)별 리스팅 규모
cnt = df["region"].value_counts()
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(cnt.index[::-1], cnt.values[::-1], color=C["main"])
ax.set_xlabel("리스팅 수")
ax.set_title(f"도시별 공급 규모 (총 {len(df):,}건, {len(cnt)}개 도시)", loc="left", fontweight="bold")
for y, v in enumerate(cnt.values[::-1]):
    ax.text(v + 800, y, f"{v:,}", va="center", fontsize=7)
plt.tight_layout(); plt.show()

print(df.groupby("country").agg(리스팅수=("id", "size"), 도시수=("region", "nunique"))
        .sort_values("리스팅수", ascending=False).to_string())

In [ ]:
# [4-2] source(0/1)의 정체 규명 — 두 집단의 성격 비교
src = df.groupby("source").agg(
    리스팅수         = ("id", "size"),
    가동_불가_비율   = ("availability_365", lambda s: (s == 0).mean()),
    avail365_중앙값  = ("availability_365", "median"),
    price_결측률     = ("price", lambda s: s.isna().mean()),
    bathrooms_결측률 = ("bathrooms", lambda s: s.isna().mean()),
    beds_결측률      = ("beds", lambda s: s.isna().mean()),
    리뷰0_비율       = ("number_of_reviews", lambda s: (s == 0).mean()),
    리뷰수_중앙값    = ("number_of_reviews", "median"),
).round(3)
print(src.T.to_string())

h0 = set(df.loc[df.source == 0, "host_id"]); h1 = set(df.loc[df.source == 1, "host_id"])
print(f"\nsource=0 호스트 {len(h0):,}명 / source=1 호스트 {len(h1):,}명 / 겹치는 호스트 {len(h0 & h1):,}명")

fig, ax = plt.subplots(figsize=(9, 3.2))
sh = df.groupby("region")["source"].mean().sort_values()
ax.bar(range(len(sh)), sh.values * 100, color=C["accent"])
ax.set_xticks(range(len(sh))); ax.set_xticklabels(sh.index, rotation=90, fontsize=7)
ax.set_ylabel("source=1 비율 (%)")
ax.set_title("도시별 source=1 비중 — 0%(아테네)부터 48%(베를린)까지 편차가 큼", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()
# --- 이월 시점 추정: last_review 로 역추적 ---
_lr = pd.to_datetime(df["last_review"], errors="coerce")
print("\n■ 이월 시점 추정 (last_review 분포)")
for s in [0, 1]:
    g = _lr[df["source"] == s]
    print(f"   source={s}: 중앙값 {g.median().date()} / 95% {g.quantile(.95).date()} / 최댓값 {g.max().date()}")

# --- 결정적 증거: 같은 호스트의 매물끼리 비교 ---
both = set(df.loc[df.source == 0, "host_id"]) & set(df.loc[df.source == 1, "host_id"])
sub  = df[df["host_id"].isin(both)]
print(f"\n■ 양쪽에 모두 나오는 호스트 {len(both):,}명의 매물만 비교 (운영 주체가 같음)")
print(sub.groupby("source").agg(
    매물수            = ("id", "size"),
    avail365_0_비율   = ("availability_365", lambda s: (s == 0).mean()),
    누적리뷰0_비율    = ("number_of_reviews", lambda s: (s == 0).mean()),
).round(3).to_string())
print("   → 같은 호스트인데 달력만 50%p 차이 = 운영 차이가 아니라 '이월로 인한 정보 부재'")

# --- availability_365 == 0 의 의미가 source마다 다른가 ---
print("\n■ availability_365 == 0 인 매물의 실제 활동 여부")
z = df[df["availability_365"] == 0]
print(z.groupby("source").agg(
    건수          = ("id", "size"),
    최근1년_거래  = ("number_of_reviews_ltm", lambda s: (s > 0).mean()),
    최근30일_거래 = ("number_of_reviews_l30d", lambda s: (s > 0).mean()),
).round(3).to_string())
print("   → source=0의 0은 '꽉 차서 0'(만실), source=1의 0은 '못 가져와서 0'(결측)")

### `source` 는 **데이터 갱신 여부**다 — `0` = 새로 수집 / `1` = 과거값 이월

`source=1`(19.4%, 163,758건)은 **"다른 종류의 매물"이 아니라 "현재 상태를 알 수 없는 매물"** 이다.
이번 수집에서 갱신에 실패해 과거 스냅샷 값이 그대로 남은 행이고, 관찰되는 차이가 전부 여기서 설명된다.

| 관찰 | 이월(carry-over)로 설명하면 |
|---|---|
| `price` 79% 결측 | 가격은 **실시간 견적**이라 이월 불가 |
| `bathrooms`·`beds` 100% 결측 | 이월된 과거 스냅샷에 해당 필드가 없었음 |
| `availability_365` 68%가 0 | 달력을 이월하지 못해 **0으로 채워진 것** — "닫힌 달력"이 아니라 **결측** |
| `last_review` 중앙값 2024-09-27 (source=0은 2026-05-25) | 이월 시점이 행마다 다르며 중앙값 기준 **약 20개월 전** |

**결정적 증거** — 양쪽에 모두 나오는 호스트 15,914명만 떼어 보면, **같은 호스트인데**
`availability_365 == 0` 비율이 source=0 **0.1%** vs source=1 **50.4%** 다. 운영 차이로는 설명되지 않는다.

**컬럼별 신뢰도 (source=1 기준)**

| 컬럼 | 신뢰도 |
|---|---|
| `price` 계열, `bathrooms`, `beds` | ❌ 사용 불가 |
| `availability_*` | ❌ **0이 만실인지 미수집인지 구분 불가** |
| `number_of_reviews_ltm` / `_l30d` / `_ly` / `reviews_per_month` | ❌ 기준 시점이 오늘이 아님 |
| `number_of_reviews`, `first/last_review`, `review_scores_*` | △ 과거 시점 값 (하한) |
| `region`, `room_type`, `property_type`, `accommodates` | ✅ 사용 가능 |

> ⚠️ **source=1 비중이 아테네 0% ~ 베를린 48%로 도시마다 다릅니다.**
> 통제 없이 도시를 비교하면 *"베를린은 가동률이 낮다"* 는 결론이 순전히 **갱신 실패율 차이**로 만들어집니다.
> **시점이 개입하는 모든 분석(가격·가동률·최근 수요)은 source=0으로 한정**하세요.

## 5. 호스트 — 경력·슈퍼호스트·포트폴리오

In [ ]:
# [5-1] 호스트 경력 — 연/월 분리 컬럼을 하나로 합쳐 사용
df["host_tenure_years"] = df["hosts_time_as_host_years"] + df["hosts_time_as_host_months"] / 12
df["user_tenure_years"] = df["hosts_time_as_user_years"] + df["hosts_time_as_user_months"] / 12

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(df["host_tenure_years"].dropna(), bins=40, color=C["main"])
axes[0].set_title("호스팅 경력(년)", loc="left", fontweight="bold"); axes[0].set_xlabel("년")
axes[1].hist((df["user_tenure_years"] - df["host_tenure_years"]).dropna(), bins=40, color=C["accent"])
axes[1].set_title("가입 → 호스팅 시작까지 걸린 기간(년)", loc="left", fontweight="bold"); axes[1].set_xlabel("년")
plt.tight_layout(); plt.show()

print(df[["user_tenure_years", "host_tenure_years"]].describe().round(2).to_string())
print("\n논리 위반(호스팅 경력 > 가입 경력):",
      (df["host_tenure_years"] > df["user_tenure_years"] + 1e-9).sum(), "건")

In [ ]:
# [5-2] 호스트 포트폴리오 규모 — 개인 호스트 vs 전문 사업자
LC = ["calculated_host_listings_count_entire_homes",
      "calculated_host_listings_count_private_rooms",
      "calculated_host_listings_count_shared_rooms"]
df["host_listings_total"] = df[LC].sum(axis=1)

bins = [0, 1, 2, 5, 20, 100, np.inf]
labs = ["1채(개인)", "2채", "3-5채", "6-20채", "21-100채", "100채+"]
df["host_scale"] = pd.cut(df["host_listings_total"], bins=bins, labels=labs)

tab = df.groupby("host_scale", observed=True).agg(
    리스팅수        = ("id", "size"),
    슈퍼호스트비율  = ("host_is_superhost", lambda s: (s == "t").mean()),
    price_중앙값    = ("price", "median"),
    avail365_중앙값 = ("availability_365", "median"),
    리뷰_ltm_중앙값 = ("number_of_reviews_ltm", "median"),
).round(3)
tab.insert(1, "비중", (tab["리스팅수"] / len(df)).round(3))
print(tab.to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].bar(tab.index.astype(str), tab["리스팅수"], color=C["main"])
axes[0].set_title("호스트 규모별 리스팅 수", loc="left", fontweight="bold")
axes[0].tick_params(axis="x", rotation=30)
axes[1].bar(tab.index.astype(str), tab["슈퍼호스트비율"] * 100, color=C["ok"])
axes[1].set_ylabel("%"); axes[1].set_title("호스트 규모별 슈퍼호스트 비율", loc="left", fontweight="bold")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

print(f"\n슈퍼호스트 비율(전체): {(df['host_is_superhost'] == 't').mean():.1%}")
print("host_listings_total == 0 인 이상 행:", (df["host_listings_total"] == 0).sum(), "건")

**해석**- 전체의 **40.6%(342,013건)가 1채 개인 호스트**, 반대편 끝에 **100채 이상 사업자가 48,701건(5.8%)** 있습니다.- **슈퍼호스트 비율이 2~20채 구간에서 35~36%로 가장 높고**, 1채(21%)와 100채+(16%)에서 낮습니다.  → "어느 정도 규모가 있으면서 직접 관리 가능한 중형 호스트"가 서비스 품질이 가장 좋다는 뜻으로, README의  *"호스트는 단일 집단이 아님"* 가설을 뒷받침하는 1차 근거입니다.- `host_listings_total == 0` 인 행이 존재합니다. 호텔 객실처럼 3개 카운트 컬럼 어디에도 잡히지 않는 유형이므로,  호스트 규모 분석 시 이 행은 **별도 'unknown' 버킷**으로 빼야 합니다(0채인데 매물이 있다는 건 모순).

## 6. 숙소 속성

In [ ]:
# [6-1] room_type / property_type / accommodates
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

rt = df["room_type"].value_counts()
axes[0].barh(rt.index[::-1], rt.values[::-1], color=C["main"])
axes[0].set_title("room_type", loc="left", fontweight="bold")
for y, v in enumerate(rt.values[::-1]):
    axes[0].text(v, y, f" {v/len(df):.1%}", va="center", fontsize=8)

pt = df["property_type"].value_counts().head(10)
axes[1].barh(pt.index[::-1], pt.values[::-1], color=C["accent"])
axes[1].set_title(f"property_type 상위 10 (총 {df['property_type'].nunique()}종)", loc="left", fontweight="bold")

ac = df["accommodates"].value_counts().sort_index()
axes[2].bar(ac.index, ac.values, color=C["ok"])
axes[2].set_xlabel("수용 인원"); axes[2].set_title("accommodates", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

print("room_type × 'hotel' 계열 property_type 정합성 점검")
hotel_pt = df["property_type"].str.contains("hotel", case=False, na=False)
print(pd.crosstab(hotel_pt, df["room_type"], rownames=["property_type에 'hotel' 포함"]).to_string())

In [ ]:
# [6-2] bathrooms / bedrooms / beds — 분포와 물리적으로 불가능한 값
spec = ["bathrooms", "bedrooms", "beds"]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, c in zip(axes, spec):
    s = df[c].dropna()
    ax.hist(s.clip(upper=s.quantile(0.999)), bins=40, color=C["main"])
    ax.set_title(f"{c}  (결측 {df[c].isna().mean():.1%})", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

print(df[spec].describe(percentiles=[.5, .95, .99]).round(2).to_string())
print("\n■ 도메인 규칙 위반 점검")
print(f"  bedrooms == 0            : {(df['bedrooms'] == 0).sum():,}건  (스튜디오/원룸일 수 있어 정상 가능)")
print(f"  beds < 1                 : {(df['beds'] < 1).sum():,}건")
print(f"  beds > 20                : {(df['beds'] > 20).sum():,}건  (최대 {df['beds'].max():.0f})")
print(f"  bathrooms > 10           : {(df['bathrooms'] > 10).sum():,}건  (최대 {df['bathrooms'].max():.0f})")
print(f"  bedrooms > 20            : {(df['bedrooms'] > 20).sum():,}건  (최대 {df['bedrooms'].max():.0f})")
print(f"  accommodates > beds*3    : {(df['accommodates'] > df['beds'] * 3).sum():,}건  (침대 대비 과도한 수용인원)")

In [ ]:
# [6-3] amenities — 개수 분포와 상위 편의시설
df["amenity_count"] = df["amenities"].str.count('",') + 1
df.loc[df["amenities"].str.len() <= 2, "amenity_count"] = 0

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].hist(df["amenity_count"], bins=60, color=C["main"])
axes[0].set_xlabel("편의시설 개수"); axes[0].set_title("amenity_count 분포", loc="left", fontweight="bold")

SAMPLE = 150_000   # 전수 파싱은 무겁기 때문에 랜덤 표본으로 상위 항목만 확인
s = df["amenities"].sample(SAMPLE, random_state=42)
top = (s.str.strip("[]").str.split('", "').explode().str.strip('" ')
        .replace("", np.nan).dropna().value_counts().head(15))
axes[1].barh(top.index[::-1], (top.values[::-1] / SAMPLE) * 100, color=C["accent"])
axes[1].set_xlabel("보유 비율 (%)")
axes[1].set_title(f"보유율 상위 편의시설 (표본 {SAMPLE:,}건)", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

print("amenity_count 요약:")
print(df["amenity_count"].describe(percentiles=[.05, .5, .95]).round(1).to_string())
print("\n빈 배열 '[]' (편의시설 0개):", (df["amenity_count"] == 0).sum(), "건 → 미등록으로 보는 편이 안전")

**해석**- `room_type` 은 **Entire home/apt 80.1%** 로 압도적입니다. 다만 `property_type` 에 "hotel"이 들어간 매물은  24,000여 건인데 `room_type == "Hotel room"` 은 3,632건뿐 — **두 컬럼이 서로 다른 기준으로 라벨링**되어 있으니  호텔 세그먼트를 볼 때는 `property_type` 기준을 쓰는 게 안전합니다.- `bathrooms 50개`, `beds 167개`, `bedrooms 50개` 같은 값은 개별 숙소로는 비현실적입니다.  대부분 **건물 전체(호스텔·아파트 동)를 한 리스팅으로 올린 케이스**로 보이며, 삭제보다는  **`flag_*` 로 분리**해 두고 객단가 분석에서 제외하는 편이 정보 손실이 적습니다.- `amenities` 는 결측이 0이지만 **빈 배열 878건은 사실상 결측**입니다(`amenity_count == 0`).

## 7. 가격 — 3종 가격 컬럼의 관계 규명

In [ ]:
# [7-1] 가격 3종 컬럼이 서로 어떤 관계인지 먼저 증명한다 (이걸 모르면 전부 오분석)
ci = pd.to_datetime(df["price_quote_checkin_date"],  errors="coerce")
co = pd.to_datetime(df["price_quote_checkout_date"], errors="coerce")
df["quote_los"] = (co - ci).dt.days          # 견적 숙박일수 Length of Stay

q = df[df["price"].notna() & df["price_quote_total_price"].notna()].copy()
q["implied_nightly"] = q["price_quote_total_price"] / q["quote_los"]

print("① price 와 (총액 ÷ 숙박일수) 의 비율 — 1.0이면 완전 일치")
print((q["price"] / q["implied_nightly"]).describe(percentiles=[.01, .5, .99]).round(4).to_string())
print("\n② price / list_price 비율 (1 - 할인율)")
print((q["price"] / q["list_price"]).describe(percentiles=[.01, .05, .5, .99]).round(3).to_string())
print(f"\n   price == list_price (할인 없음) : {(abs(q.price - q.list_price) < 1e-9).mean():.1%}")
print(f"   price <  list_price (할인 있음) : {(q.price < q.list_price).mean():.1%}")
print(f"   price >  list_price (모순)      : {(q.price > q.list_price + 1e-9).sum():,}건")

print("\n③ 견적 숙박일수(LOS) 분포 — 총액을 그대로 비교하면 안 되는 이유")
print(df["quote_los"].value_counts().head(8).to_string())
print(f"   LOS 최소 {df['quote_los'].min():.0f}박 ~ 최대 {df['quote_los'].max():.0f}박")

In [ ]:
# [7-2] discount_type 이 실제 할인과 일치하는지 검증
d2 = df.assign(할인여부=(df["price"] < df["list_price"]),
               할인율=(1 - df["price"] / df["list_price"]),
               discount=df["discount_type"].fillna("(없음)"))
tab = d2.groupby("discount").agg(건수=("id", "size"),
                                 실제할인비율=("할인여부", "mean"),
                                 평균할인율=("할인율", "mean")).round(3)
print(tab.to_string())

sub = d2[d2["할인여부"]]
fig, ax = plt.subplots(figsize=(7, 3.2))
rate = sub["할인율"] * 100
ax.hist(rate, bins=50, color=C["accent"])
ax.set_xlabel("할인율 (%)"); ax.set_ylabel("건수")
ax.set_title(f"실제 할인이 적용된 {len(sub):,}건의 할인율 분포 (중앙값 {rate.median():.1f}%)",
             loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# [7-3] price 분포와 통화 단위 점검 (국가별 중앙값이 현실적인가?)
p = df["price"].dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].hist(p.clip(upper=1000), bins=80, color=C["main"])
axes[0].set_xlabel("price (1000 이상은 1000으로 클리핑)")
axes[0].set_title("price 원 스케일 — 극단적 우편향", loc="left", fontweight="bold")
axes[1].hist(np.log10(p[p > 0]), bins=80, color=C["ok"])
axes[1].set_xlabel("log10(price)")
axes[1].set_title("price 로그 스케일 — 로그정규에 가까움", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

print(p.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2).to_string())
print("\n국가별 1박 가격 중앙값 (통화 단위 점검용)")
print(df.groupby("country")["price"].agg(중앙값="median", 평균="mean", 건수="count")
        .round(1).sort_values("중앙값").to_string())

In [ ]:
# [7-4] price 이상치 — 절대 기준 + 1인당 가격 기준
df["price_per_person"] = df["price"] / df["accommodates"]
print("■ 절대 기준")
for m, label in [(df["price"] < 1, "price < 1"), (df["price"] < 10, "price < 10"),
                 (df["price"] > 2000, "price > 2,000"), (df["price"] > 10000, "price > 10,000")]:
    print(f"  {label:18s}: {m.sum():>7,}건 ({m.mean():.3%})")

print("\n■ 최고가 상위 5건 — 대부분 통화 미환산 또는 입력 오류로 의심")
print(df.nlargest(5, "price")[["region", "room_type", "accommodates", "price", "list_price",
                               "minimum_nights", "number_of_reviews"]].to_string())
print("\n■ 최저가 하위 5건 — 이스탄불 집중 + minimum_nights 365 → '노출만 걸어둔' 더미 매물 패턴")
print(df.nsmallest(5, "price")[["region", "room_type", "accommodates", "price",
                                "minimum_nights", "number_of_reviews", "availability_365"]].to_string())

fig, ax = plt.subplots(figsize=(10, 4))
order = df.groupby("region")["price"].median().sort_values().index
data = [np.log10(df.loc[df.region == r, "price"].dropna().clip(lower=0.1)) for r in order]
bp = ax.boxplot(data, showfliers=False, patch_artist=True)
for b in bp["boxes"]:
    b.set(facecolor=C["main"], alpha=0.6)
ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(order, rotation=90, fontsize=7)
ax.set_ylabel("log10(1박 가격)")
ax.set_title("도시별 1박 가격 분포 (로그) — 이상치 제외 표시", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

### 가격 컬럼 3종의 정체 — 검증 결과```price = price_quote_total_price / quote_los        (1박 실판매가)price = list_price × (1 - 할인율)```> ⚠️ **`price`는 "순수 1박 숙박료"가 아니다.** `price_quote_total_price` 는 공식 정의상 **청소비·수수료를 포함한 총액**이고,
> `price` 는 그 총액을 박수로 나눈 값이다(잔차 중앙값 0, 99%가 ±0.14). 즉 **수수료 포함 1박 환산 단가**다.
> 청소비가 숙박일수로 나뉘므로 **짧은 견적일수록 `price`가 부풀려진다.** 세그먼트 비교에는 문제없지만
> "이 숙소의 1박 요금"으로 단정하지는 말 것.

- ①의 비율이 전 구간에서 **1.000** (최소 0.985 · 최대 1.009, 반올림 오차 수준) → `price` 는 **견적 총액을 숙박일수로 나눈 1박 실판매가**.- `price ≤ list_price` 가 100% 성립, `price < list_price` 인 **13.8%가 `discount_type` 5종과 정확히 1:1 대응**  → `list_price` = **할인 전 정가**, `discount_type` 결측 = **할인 없음**.- **⚠️ `price_quote_total_price` 를 그대로 객단가로 쓰면 안 됩니다.** 견적 LOS가 1박(23만 건)부터 100박(9천 건)까지 섞여 있어,  총액 비교는 "가격이 비싼 매물"이 아니라 "견적 기간이 긴 매물"을 고르게 됩니다. **반드시 `price`(1박가)를 사용**하세요.**통화** — 튀르키예 54€, 네덜란드 268€ 로 국가별 중앙값이 모두 현실적인 유로 범위입니다. → **EUR로 환산 완료된 값**으로 판단.단 상위 극단값(런던 611,870)은 환산 누락·입력 오류가 섞여 있습니다.**이상치 성격이 두 가지로 나뉩니다.**- **상단(price > 2,000, 3,799건 · 0.55%)**: 통화 오류/총액 혼입 → **분석에서 제외 대상**.- **하단(price < 10, 4,154건)**: 이스탄불에 집중되며 `minimum_nights = 365`, 리뷰 0건 —  삭제 대상이 아니라 **"검색 노출만 노린 유령 매물"이라는 발견 그 자체**입니다. 수익성 진단의 소재로 살려 두세요.

## 8. 숙박일 제약 — `minimum_nights` / `maximum_nights`

In [ ]:
# [8-1] 최소/최대 숙박일 분포와 인위적 집중값(센티넬)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].hist(df["minimum_nights"].dropna().clip(upper=40), bins=40, color=C["main"])
axes[0].set_xlabel("minimum_nights (40 이상 클리핑)")
axes[0].set_title("minimum_nights", loc="left", fontweight="bold")

mx = df["maximum_nights"].dropna()
axes[1].hist(np.log10(mx.clip(lower=1)), bins=60, color=C["accent"])
axes[1].set_xlabel("log10(maximum_nights)")
axes[1].set_title("maximum_nights — 1125 / 365 에 인위적 집중", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

print("minimum_nights 상위 빈도값");  print(df["minimum_nights"].value_counts().head(8).to_string())
print("\nmaximum_nights 상위 빈도값"); print(df["maximum_nights"].value_counts().head(8).to_string())
print(f"\nmaximum_nights >= 1125 (사실상 '제한 없음') : {(df['maximum_nights'] >= 1125).mean():.1%}")
print(f"maximum_nights == 2147483647 (int32 센티넬) : {(df['maximum_nights'] == 2147483647).sum():,}건")
print(f"minimum_nights > maximum_nights (논리 모순)  : {(df['minimum_nights'] > df['maximum_nights']).sum():,}건")
print(f"minimum_nights >= 30 (사실상 장기임대)       : {(df['minimum_nights'] >= 30).mean():.1%}")
print(f"minimum_nights > 365                         : {(df['minimum_nights'] > 365).sum():,}건")

In [ ]:
# [8-2] 최소 숙박일이 가동률/객단가에 미치는 영향 (source=0, 가격 있는 매물만)
base = df[(df["source"] == 0) & df["price"].notna()].copy()
base["occ90"] = 1 - base["availability_90"] / 90     # 향후 90일 가동률 프록시

mb = pd.cut(base["minimum_nights"], [0, 1, 2, 3, 7, 29, np.inf],
            labels=["1박", "2박", "3박", "4-7박", "8-29박", "30박+"])
t = base.groupby(mb, observed=True).agg(
    건수          = ("id", "size"),
    가동률_90일   = ("occ90", "mean"),
    price_중앙값  = ("price", "median"),
    리뷰_ltm_중앙 = ("number_of_reviews_ltm", "median"),
).round(3)
print(t.to_string())

fig, ax1 = plt.subplots(figsize=(8, 3.4))
x = np.arange(len(t))
ax1.bar(x, t["가동률_90일"] * 100, color=C["main"])
ax1.set_ylabel("가동률 (%)", color=C["main"])
ax1.set_xticks(x); ax1.set_xticklabels(t.index.astype(str))
ax2 = ax1.twinx(); ax2.grid(False)
ax2.plot(x, t["price_중앙값"], color=C["bad"], marker="o", lw=2)
ax2.set_ylabel("price 중앙값 (€)", color=C["bad"])
ax1.set_title("최소 숙박일 × 가동률(막대) × 객단가(선) — 4-7박에서 정점, 30박+에서 급락",
              loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

**해석 (README의 "`minimum_nights` 가 예약 전환에 미치는 영향" 에 대한 직접적 근거)**- `maximum_nights` 의 **33.8%가 1125** 에 몰려 있습니다. 이는 플랫폼 기본값(≈ 3년)이며 **"호스트가 설정하지 않았다"는 뜻**입니다.  실제 제약이 아니므로 그대로 쓰면 왜곡되고, `max_nights_is_default` 같은 플래그로 다루는 게 맞습니다.- `minimum_nights > maximum_nights` 인 **논리 모순 48건**이 있습니다(소수라 제거해도 무방).- **가동률은 1박(44.2%) → 4-7박(54.6%)까지 올라갔다가 30박+(32.2%)에서 급락**하고,  객단가도 4-7박(205€)이 최고, 30박+(88€)이 최저입니다.  → 최소 숙박일 30일 이상(전체 9.0%)은 **가동률·객단가 모두 최악**인 구간으로, 규제 회피용 장기임대 전환 매물로 추정됩니다.  이 세그먼트는 수수료 수익 기여가 거의 없는 **"비용 요인"** 입니다.

## 9. 예약 가능성 → 가동률 프록시

In [ ]:
# [9-1] availability 4종 분포와 양극단 집중
av = ["availability_30", "availability_60", "availability_90", "availability_365"]
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, c in zip(axes, av):
    ax.hist(df[c], bins=40, color=C["main"])
    ax.set_title(f"{c}\n0인 비율 {df[c].eq(0).mean():.1%} / 최대값 비율 {df[c].eq(df[c].max()).mean():.1%}",
                 loc="left", fontsize=9, fontweight="bold")
plt.tight_layout(); plt.show()

print(df[av].describe(percentiles=[.25, .5, .75]).round(1).to_string())
print("\n■ 단조성 검증 (a30 ≤ a60 ≤ a90 ≤ a365 이어야 함)")
print("  a30 > a60 :", (df.availability_30 > df.availability_60).sum())
print("  a60 > a90 :", (df.availability_60 > df.availability_90).sum())
print("  a90 > a365:", (df.availability_90 > df.availability_365).sum(), "→ 위반 없음, 내부 정합성 양호")

In [ ]:
# [9-2] '죽은 재고' 규모 + 도시별 가동률 × 객단가
dead = (df["availability_365"] == 0) & (df["number_of_reviews"] == 0)
print(f"availability_365 == 0      : {df['availability_365'].eq(0).sum():>7,}건 ({df['availability_365'].eq(0).mean():.1%})")
print(f"number_of_reviews == 0     : {df['number_of_reviews'].eq(0).sum():>7,}건 ({df['number_of_reviews'].eq(0).mean():.1%})")
print(f"number_of_reviews_ltm == 0 : {df['number_of_reviews_ltm'].eq(0).sum():>7,}건 ({df['number_of_reviews_ltm'].eq(0).mean():.1%})")
print(f"둘 다 0 (완전 비활성 재고)  : {dead.sum():>7,}건 ({dead.mean():.1%})")

# 도시별 수익성 프록시 = 1박가 × 연간 예약일수 (source=0, 가격 보유분)
base["est_annual_rev"] = base["price"] * (365 - base["availability_365"])
r = base.groupby("region").agg(
    건수              = ("id", "size"),
    price_중앙값      = ("price", "median"),
    가동률_365        = ("availability_365", lambda s: (1 - s / 365).mean()),
    추정연매출_중앙값 = ("est_annual_rev", "median"),
).round(2).sort_values("추정연매출_중앙값", ascending=False)

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.scatter(r["가동률_365"] * 100, r["price_중앙값"], s=np.sqrt(r["건수"]) * 1.6,
           color=C["main"], alpha=0.6, edgecolor="white")
for name, row in r.iterrows():
    if row["추정연매출_중앙값"] > 30000 or row["추정연매출_중앙값"] < 11000 or row["건수"] > 40000:
        ax.annotate(name, (row["가동률_365"] * 100, row["price_중앙값"]),
                    fontsize=7.5, xytext=(4, 3), textcoords="offset points")
ax.set_xlabel("평균 가동률 (%)"); ax.set_ylabel("1박 가격 중앙값 (€)")
ax.set_title("도시별 가동률 × 객단가 (원 크기 = 공급 규모)\n오른쪽 위일수록 숙소당 수익성이 높음",
             loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

print("\n[상위 8]"); print(r.head(8).to_string())
print("\n[하위 5]"); print(r.tail(5).to_string())
# --- 죽은 재고는 source=0 기준으로만 세야 한다 ---
print("\n■ 죽은 재고: 전체 vs source=0 (이월분 제외)")
_ref = pd.to_datetime(df["last_review"], errors="coerce").max()
rows = {}
for nm, g in [("전체", df), ("source=0", df[df.source == 0])]:
    st = (_ref - pd.to_datetime(g["last_review"], errors="coerce")).dt.days
    rows[nm] = {
        "n":                    len(g),
        "최근12개월_리뷰0":     g["number_of_reviews_ltm"].eq(0).mean(),
        "누적리뷰0":            g["number_of_reviews"].eq(0).mean(),
        "avail365_0":           g["availability_365"].eq(0).mean(),
        "1년이상_리뷰없음":     st.gt(365).mean(),
    }
print(pd.DataFrame(rows).T.round(3).to_string())
print(f"\n   avail365==0 중 source=1 비중: {df.loc[df.availability_365 == 0, 'source'].mean():.1%}"
      "  → 이 지표는 죽은 재고 근거로 쓸 수 없음")
print(f"   ltm==0 중 source=1 비중     : {df.loc[df.number_of_reviews_ltm == 0, 'source'].mean():.1%}")
print(f"   ✅ 유효 수치: source=0 기준 최근 12개월 리뷰 0건 "
      f"{df[(df.source == 0) & (df.number_of_reviews_ltm == 0)].shape[0]:,}건")

**해석**
- `availability_*` 는 **4종이 완벽히 단조** (a30 ≤ a60 ≤ a90 ≤ a365, 위반 0건) → 내부 정합성은 양호.
- 🚨 **그러나 `availability_365 == 0` 인 112,397건 중 99.5%(111,838건)가 source=1**, 즉 달력을 이월하지 못해
  0으로 채워진 **결측**이다. **source=0만 보면 559건(0.1%)뿐**이고, 그나마 **70.1%가 최근 1년 내 리뷰 보유** →
  "닫힌 달력"이 아니라 **"꽉 차서 0"(만실)** 이다. **같은 0이 두 source에서 정반대 의미**다.
- 따라서 **죽은 재고의 근거로는 `number_of_reviews_ltm == 0` 하나만 쓴다.**
  전체 기준 38.1%지만 source=1의 이월분이 섞여 있어, **유효한 수치는 source=0 기준 207,005건(30.5%)** 이다.
- 산점도는 README의 *"도시 규모 ≠ 수익성"* 을 직접 보여준다.
  **런던(59,345건)·시칠리아(50,288건)는 공급 최상위인데 추정 연매출 중앙값이 2.2만€·1.2만€** 인 반면,
  **마요르카·메노르카·암스테르담은 공급이 훨씬 적은데도 5~7만€** 로 3~5배 높다.
  **이스탄불은 가동률 20%·객단가 81€로 최하위(2,060€).**

> ⚠️ `est_annual_rev = price × (365 - availability_365)` 는 **프록시**다. "비어 있지 않음"에는 예약뿐 아니라
> 호스트가 막아둔 날(blocked)도 포함되므로 과대추정된다. 절대값이 아니라 **도시 간 상대 비교**로만 쓸 것.

## 10. 리뷰 — 수요·품질 지표

In [ ]:
# [10-1] 리뷰 수와 기간
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].hist(np.log10(df["number_of_reviews"] + 1), bins=50, color=C["main"])
axes[0].set_xlabel("log10(누적 리뷰 수 + 1)")
axes[0].set_title(f"number_of_reviews (0건 {df['number_of_reviews'].eq(0).mean():.1%})",
                  loc="left", fontweight="bold")

first = pd.to_datetime(df["first_review"], errors="coerce")
last  = pd.to_datetime(df["last_review"],  errors="coerce")
axes[1].hist(first.dropna(), bins=60, color=C["ok"])
axes[1].set_title("first_review — 신규 공급 유입 추이", loc="left", fontweight="bold")
axes[1].tick_params(axis="x", rotation=30)

REF = last.max()                        # 데이터 기준일
stale = (REF - last).dt.days
axes[2].hist(stale.dropna().clip(upper=1500), bins=60, color=C["bad"])
axes[2].set_xlabel("마지막 리뷰 이후 경과일 (1500일 클리핑)")
axes[2].set_title("영업 활성도", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

print("데이터 기준일(last_review 최댓값):", REF.date())
print(f"first_review 범위: {first.min().date()} ~ {first.max().date()}")
print(f"last_review < first_review (논리 모순): {(last < first).sum()}건")
print(f"\n리뷰 보유 매물 중 1년 이상 리뷰 없음: {(stale > 365).mean():.1%}")
print(stale.describe(percentiles=[.25, .5, .75, .9]).round(0).to_string())
# --- 공식 정의 검증 ① _ltm 과 _ly 는 겹치는가 ---
print("\n■ number_of_reviews_ltm vs _ly")
print(f"   완전 일치 비율      : {(df.number_of_reviews_ltm == df.number_of_reviews_ly).mean():.1%}")
print(f"   상관계수            : {df.number_of_reviews_ltm.corr(df.number_of_reviews_ly):.4f}")
print(f"   _ltm + _ly > 누적리뷰: {(df.number_of_reviews_ltm + df.number_of_reviews_ly > df.number_of_reviews).mean():.1%}"
      "  ← 두 기간이 겹친다는 직접 증거")
print("   → 전년 대비 성장률(_ly 대비 _ltm) 계산에 쓰면 안 됨")

# --- 공식 정의 검증 ② reviews_per_month 는 '전체 기간' 평균인가 ---
_g = df[df.reviews_per_month.notna() & first.notna()]
_months = (REF - first[_g.index]).dt.days / 30.44
_calc = _g.number_of_reviews / _months
print("\n■ reviews_per_month = 누적 리뷰 ÷ (최초 리뷰~기준일 개월수) 인가?")
print(f"   상관계수 {_g.reviews_per_month.corr(_calc):.4f} / 비율 중앙값 {(_g.reviews_per_month / _calc).median():.3f}")
print("   → 최근 활동량이 아니라 '생애 평균'. 최근 수요는 number_of_reviews_ltm 을 쓸 것")

In [ ]:
# [10-2] 평점 7종 — 분포와 상호 상관
sc = [c for c in df.columns if c.startswith("review_scores_")]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

axes[0].boxplot([df[c].dropna() for c in sc], showfliers=False, patch_artist=True,
                boxprops=dict(facecolor=C["main"], alpha=0.6))
axes[0].set_xticks(range(1, len(sc) + 1))
axes[0].set_xticklabels([c.replace("review_scores_", "") for c in sc], rotation=30, fontsize=8)
axes[0].set_title("평점 7종 분포 — 천장효과(대부분 4.5~5.0)", loc="left", fontweight="bold")

corr = df[sc].corr()
axes[1].imshow(corr, cmap="Blues", vmin=0.3, vmax=1)
axes[1].set_xticks(range(len(sc)))
axes[1].set_xticklabels([c.replace("review_scores_", "") for c in sc], rotation=60, ha="right", fontsize=8)
axes[1].set_yticks(range(len(sc)))
axes[1].set_yticklabels([c.replace("review_scores_", "") for c in sc], fontsize=8)
for i in range(len(sc)):
    for j in range(len(sc)):
        axes[1].text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=7,
                     color="white" if corr.values[i, j] > 0.75 else "black")
axes[1].grid(False); axes[1].set_title("평점 간 상관", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

print(df[sc].describe(percentiles=[.05, .25, .5]).round(2).to_string())
print("\n평점이 정확히 0인 행 (사실상 무효값):")
print((df[sc] == 0).sum().to_string())

**해석**
- 🚨 **`number_of_reviews_ltm` 과 `_ly` 는 서로 다른 해가 아니다.** 둘이 **46.2% 완전 일치**, 상관 0.93이고
  `_ltm + _ly > number_of_reviews` 인 행이 **17.7%** — 두 기간이 **겹친다는 직접 증거**다.
  **`_ly / _ltm` 로 전년 대비 성장률을 구하면 안 된다.**
- 🚨 **`reviews_per_month` 는 최근 활동량이 아니라 "최초 리뷰 이후 전체 기간"의 생애 평균이다.**
  실측으로도 `누적 리뷰 ÷ (최초 리뷰~기준일 개월수)` 와 **상관 0.991 · 비율 중앙값 1.009** 로 일치한다.
  2019년에 활발했다 지금 죽은 매물도 이 값은 높다. **최근 수요는 `number_of_reviews_ltm`(source=0 한정)** 을 쓸 것.
- 평점은 **천장효과**가 뚜렷하다. `review_scores_rating` 중앙값 4.85, 75분위가 이미 5.0 → **높은 점수로는 변별이 안 된다.**
  변별력은 하위 꼬리(4.0 이하)와, 7종 중 **평균이 가장 낮은 `value`(가성비, 4.65)** 및 `cleanliness`(4.73)에 있다.
- 평점 7종은 서로 상관이 높아(0.7~0.9) **전부 넣으면 다중공선성**이 생긴다. `rating` + `value` + `cleanliness` 정도만 권장.
- **평점이 정확히 0인 행 15~35건**은 5점 척도에서 나올 수 없으므로 결측 처리 대상.
- 리뷰 보유 매물 중 1년 이상 리뷰 없음: 전체 19.5% → **source=0만 보면 13.0%**.

## 11. 이상치 종합 — 도메인 규칙 기반 플래그

In [ ]:
# [11-1] 이상치 규칙을 한 곳에 모아 플래그로 만든다 (삭제가 아니라 '표시')
REF = pd.to_datetime(df["last_review"], errors="coerce").max()
stale_days = (REF - pd.to_datetime(df["last_review"], errors="coerce")).dt.days

rules = {
    "price_초과_2000":            df["price"] > 2000,
    "price_미만_10":              df["price"] < 10,
    "1인당가격_초과_1000":        df["price_per_person"] > 1000,
    "min_nights_초과_365":        df["minimum_nights"] > 365,
    "min>max_nights_모순":        df["minimum_nights"] > df["maximum_nights"],
    "max_nights_기본값1125":      df["maximum_nights"] >= 1125,
    "beds_초과_20":               df["beds"] > 20,
    "bathrooms_초과_10":          df["bathrooms"] > 10,
    "bedrooms_초과_20":           df["bedrooms"] > 20,
    "평점_0":                     df["review_scores_rating"] == 0,
    "amenities_빈배열":           df["amenity_count"] == 0,
    "host_listings_0_모순":       df["host_listings_total"] == 0,
    "완전비활성(가동0+리뷰0)":    (df["availability_365"] == 0) & (df["number_of_reviews"] == 0),
    "1년이상_리뷰없음":           stale_days > 365,
}
summary = pd.DataFrame({"건수": {k: int(v.fillna(False).sum()) for k, v in rules.items()}})
summary["비율(%)"] = (summary["건수"] / len(df) * 100).round(3)
summary["처리권고"] = [
    "제외(통화오류 의심)", "유지+플래그(유령매물 발견)", "제외", "제외", "제외",
    "플래그만(미설정 의미)", "플래그(건물단위 리스팅)", "플래그", "플래그",
    "결측 처리", "결측 처리", "unknown 버킷", "유지(분석 대상 그 자체)", "유지(분석 대상 그 자체)",
]
print(summary.to_string())

for k, v in rules.items():
    df["flag_" + k] = v.fillna(False)

hard = ["flag_price_초과_2000", "flag_1인당가격_초과_1000",
        "flag_min_nights_초과_365", "flag_min>max_nights_모순"]
n_hard = df[hard].any(axis=1).sum()
print(f"\n'제외 권고' 규칙 중 하나라도 걸리는 행: {n_hard:,}건 ({n_hard/len(df):.2%}) → 제거해도 손실 미미")

### 이상치 처리 원칙 — **"통계적 이상치 ≠ 제거 대상"**이 데이터에 IQR 기준(Q1-1.5IQR ~ Q3+1.5IQR)을 그대로 적용하면`price` 8.3%, `minimum_nights` 13.2%, `number_of_reviews` 11.6% 가 이상치로 잡힙니다.하지만 숙박 데이터의 가격·리뷰 수는 **원래 로그정규 분포**라 IQR로 자르면 정상적인 고가 매물까지 날아갑니다.그래서 **분포 기준이 아니라 도메인 규칙 기준**으로 세 단계로 나눕니다.| 단계 | 대상 | 조치 | 이유 ||---|---|---|---|| **① 제거** | 물리적/논리적으로 불가능 — `price>2,000`, `1인당>1,000€`, `min>max`, `min_nights>365` | 행 제외 (총 **0.6% 미만**) | 데이터 오류가 확실하고 양이 적어 제거해도 편향이 안 생김 || **② 플래그** | 비현실적이지만 설명 가능 — `beds>20`, `bathrooms>10`, `max_nights=1125` | 컬럼 유지 + `flag_*` 생성 | 건물 단위 리스팅·미설정 기본값 등 **실제 현상**이므로 삭제하면 정보가 사라짐 || **③ 보존(분석 대상)** | `price<10` 유령 매물, 가동 0 + 리뷰 0, 1년 이상 리뷰 없음 | 절대 삭제 금지 | **이게 바로 찾아야 할 문제**입니다. 제거하면 과제의 결론 자체가 사라집니다 |> 특히 ③을 실수로 지우기 쉽습니다. "예약 안 되는 숙소는 매출 0 + 노출/지원 리소스 소모"라는 과제 조건상> **비활성 재고는 노이즈가 아니라 분석의 주인공**입니다.

## 12. 결측치 처리 방안 — 컬럼별 권고와 논거

In [ ]:
# [12-1] 결측 처리 권고안 요약표
plan = pd.DataFrame([
 ("price / list_price / price_quote_total_price", 18.6, "MNAR (예약 불가)",
  "대체 금지 · 별도 세그먼트 분리",
  "결측 <-> availability_365==0 (99.9%). 값이 비었다는 사실 자체가 '판매 불가'라는 정보. "
  "평균/중앙값 대체 시 죽은 재고가 정상 매물로 둔갑해 수익성이 과대추정됨"),
 ("price_quote_checkin/checkout_date", 14.9, "MNAR (견적 미산출)",
  "quote_los 파생에만 사용, 별도 대체 불필요",
  "가격 해석용 보조 컬럼. 이 값이 없으면 price 도 없으므로 독립적으로 채울 이유가 없음"),
 ("bathrooms", 25.1, "구조적 (source=1 미제공)",
  "source=0 한정 분석 (+ 필요 시 source=0 내부에서 도시×room_type×accommodates 중앙값 대체)",
  "source=1은 100% 결측이라 이 컬럼으로 source 간 비교 자체가 불가능. "
  "16만 건을 대체하면 없는 정보를 지어내는 것이므로, 욕실 수가 필요한 분석은 source=0으로 한정하는 것이 정직함"),
 ("beds", 21.7, "구조적 (source=1 미제공)",
  "위와 동일. 필요 시 accommodates 기반 규칙 대체 (beds ≈ ceil(accommodates/2))",
  "beds는 accommodates와 상관이 높아 규칙 대체의 근거가 있음. 단 대체 여부를 imputed 플래그로 반드시 남길 것"),
 ("bedrooms", 16.2, "혼합 (source 무관 결측도 존재)",
  "(도시 × property_type × accommodates) 중앙값 대체 + 플래그",
  "source=0에도 13.8% 결측이 있어 구조적이지 않음. 유사 매물 그룹 중앙값이 합리적. "
  "단 bedrooms=0(스튜디오)은 결측이 아니므로 0을 NaN으로 바꾸지 말 것"),
 ("review_scores_* 7종 / reviews_per_month / first·last_review", 19.7, "구조적 (리뷰 0건)",
  "대체 금지 · has_review 플래그로 분리 분석",
  "리뷰 0건 <-> 평점 결측이 완전 일치. 0으로 채우면 '최악 평점'이 되고 평균으로 채우면 '검증된 매물'이 되어 양쪽 다 틀림. "
  "신규/미검증 매물은 별개 세그먼트로 다뤄야 함"),
 ("discount_type", 88.9, "결측이 아니라 '해당 없음'",
  "'(없음)' 범주로 채움",
  "결측 <-> price==list_price 가 100% 대응. 결측이 아니라 '할인 미적용'이라는 명확한 상태값. 결측률이 가장 높지만 삭제하면 안 되는 대표 사례"),
 ("neighbourhood_group_cleansed", 74.7, "구조적 (도시별 스키마 차이)",
  "이 컬럼 기반 전체 분석 포기 · region 을 지역 단위로 사용",
  "53개 도시 중 36개가 100% 결측, 17개가 0% 결측으로 이분됨. 17개 도시만으로 분석하면 도시 선택 편향이 생김. "
  "지역 세분화가 꼭 필요하면 해당 17개 도시 내부 분석으로 범위를 명시적으로 한정"),
 ("host_location", 27.6, "MCAR에 가까움 (자유 입력 미기재)",
  "'Unknown' 범주로 채움 + 국가만 파싱해 사용",
  "유일하게 무작위에 가까운 결측. 13,348개 표기 변형이 있어 원문 그대로는 못 쓰고, "
  "쉼표 뒤 국가만 추출해 '현지 호스트 vs 원격 호스트' 이진 변수로 쓰는 것이 실용적"),
 ("host_is_superhost / hosts_time_as_*", 0.11, "MCAR (호스트 메타 수집 실패)",
  "행 삭제 (또는 superhost='f' 대체)",
  "900건 전부 동일한 행에서 호스트 메타 전체가 비어 있음. 0.11%에 불과해 삭제해도 편향이 생기지 않음"),
 ("minimum_nights / maximum_nights", 0.02, "MCAR",
  "행 삭제",
  "139건뿐이라 어떤 처리를 해도 결과가 바뀌지 않음. 가장 단순한 삭제가 적절"),
], columns=["컬럼", "결측률(%)", "결측 메커니즘", "권고 처리", "논거"])

pd.set_option("display.max_colwidth", 400)
plan.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap", "vertical-align": "top"}) \
          .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])

### 결측 처리의 3대 원칙 (왜 이렇게 권고하는가)**원칙 1 — 결측 메커니즘을 먼저 판정하고, 그 다음에 처리 방법을 고른다.**"결측률 몇 % 이상이면 삭제" 같은 기계적 기준은 위험합니다. 이 데이터에서`discount_type` 은 결측률이 88.9%로 가장 높지만 **삭제하면 안 되는 컬럼**입니다(결측 = 할인 없음이라는 정보).반대로 `price` 는 18.6%로 낮은 편이지만 **대체하면 안 되는 컬럼**입니다(결측 = 판매 불가).**원칙 2 — 구조적 결측(MNAR)은 채우지 말고 세그먼트를 나눈다.**`price`, `review_scores_*`, `bathrooms`/`beds` 는 모두 "값이 있을 수 없어서 비어 있는" 경우입니다.이 세 덩어리를 대체하면 842,655건이 전부 살아남지만, 그 대가로**"팔리지 않는 매물 156,439건"과 "리뷰 한 번 못 받은 매물 165,634건"이라는 가장 중요한 발견이 통계 속에 묻힙니다.**과제의 목표가 *"공급 확보 → 노출 → 예약 전환 → 재방문 흐름의 문제 찾기"* 이므로,**전환에 실패한 매물을 구분해서 살려 두는 것이 분석의 핵심**입니다.**원칙 3 — 대체를 하기로 했다면 반드시 ① 유사 그룹 기준으로, ② 대체 여부를 플래그로 남긴다.**`bedrooms` 처럼 대체가 정당한 경우에도 전체 중앙값(1개)을 일괄 적용하면 13.6만 건이 같은 값이 되어 분산이 붕괴합니다.**(도시 × property_type × accommodates)** 처럼 유사 매물 그룹 내 중앙값을 쓰고, `bedrooms_imputed = True` 플래그를 남겨나중에 "대체값이 결론을 만든 게 아닌지" 검증할 수 있게 해야 합니다.**요약 결정표**| 처리 | 대상 컬럼 | 결과 ||---|---|---|| **행 삭제** | `hosts_time_*`, `host_is_superhost`(900건), `minimum/maximum_nights`(139건) | **총 1,039건 (0.12%) 만 삭제** || **범주로 채움** | `discount_type` → `'(없음)'`, `host_location` → `'Unknown'` | 정보 손실 0 || **그룹 중앙값 대체 + 플래그** | `bedrooms` (필요 시 `beds`) | 분산 유지 || **대체 금지, 세그먼트 분리** | `price` 계열, `review_scores_*` 계열, `bathrooms`/`beds` | **분석의 핵심 발견 보존** || **컬럼 사용 포기** | `neighbourhood_group_cleansed` | `region` 으로 대체 |

## 13. 분석용 데이터셋 만들기

In [ ]:
# [13-1] 위 권고안을 그대로 적용해 분석용 데이터셋을 만든다
d = df.copy()
before = len(d)

# ── (1) 행 삭제: MCAR 소량 결측 + 논리 모순/명백한 오류 ────────────────────────
drop_mask = (
      d["host_is_superhost"].isna()                              # 호스트 메타 전체 결측 900건
    | d["minimum_nights"].isna() | d["maximum_nights"].isna()    # 139건
    | (d["minimum_nights"] > d["maximum_nights"])                # 논리 모순 48건
    | (d["minimum_nights"] > 365)                                # 156건
    | (d["price"] > 2000)                                        # 통화 오류 의심 3,799건
    | (d["price_per_person"] > 1000)
)
d = d[~drop_mask].copy()
print(f"(1) 행 삭제: {before:,} → {len(d):,}  ({(before-len(d))/before:.2%} 제거)")

# ── (2) '해당 없음'을 명시적 범주로 ──────────────────────────────────────────
d["discount_type"] = d["discount_type"].fillna("(없음)")
d["host_location"] = d["host_location"].fillna("Unknown")

# ── (3) 무효값(0점) → 결측 ───────────────────────────────────────────────────
sc = [c for c in d.columns if c.startswith("review_scores_")]
d[sc] = d[sc].mask(d[sc] == 0)

# ── (4) 대체(유사 그룹 중앙값) + 대체 플래그 ─────────────────────────────────
d["bedrooms_imputed"] = d["bedrooms"].isna()
grp = ["region", "property_type", "accommodates"]
d["bedrooms"] = d["bedrooms"].fillna(d.groupby(grp)["bedrooms"].transform("median"))
d["bedrooms"] = d["bedrooms"].fillna(d.groupby("region")["bedrooms"].transform("median"))
print(f"(4) bedrooms 대체: {d['bedrooms_imputed'].sum():,}건 → 잔여 결측 {d['bedrooms'].isna().sum():,}건")

# ── (5) 세그먼트 플래그 (대체 대신 '구분') ───────────────────────────────────
REF = pd.to_datetime(d["last_review"], errors="coerce").max()
d["has_price"]   = d["price"].notna()                      # 판매 가능 = 수익 창출 후보
d["has_review"]  = d["number_of_reviews"] > 0              # 검증된 매물
d["is_bookable"] = d["availability_365"] > 0
d["is_active"]   = d["number_of_reviews_ltm"] > 0          # 최근 1년 실거래
d["is_dormant"]  = ~d["is_active"]
d["days_since_last_review"] = (REF - pd.to_datetime(d["last_review"], errors="coerce")).dt.days

# ── (6) 분석용 파생변수 ──────────────────────────────────────────────────────
d["occ_30"]  = 1 - d["availability_30"]  / 30
d["occ_90"]  = 1 - d["availability_90"]  / 90
d["occ_365"] = 1 - d["availability_365"] / 365
d["est_annual_revenue"] = d["price"] * (365 - d["availability_365"])   # 프록시
d["platform_fee"]       = d["est_annual_revenue"] * 0.155              # 수수료 15.5%
d["discount_rate"]      = 1 - d["price"] / d["list_price"]
d["is_pro_host"]        = d["host_listings_total"] >= 6
d["max_nights_is_default"] = d["maximum_nights"] >= 1125

print("\n■ 최종 세그먼트 구성")
seg = pd.DataFrame({"건수": [
    len(d), int(d["has_price"].sum()), int(d["is_bookable"].sum()), int(d["has_review"].sum()),
    int(d["is_active"].sum()), int(d["is_dormant"].sum()), int((d["source"] == 0).sum())]},
    index=["전체", "가격 있음(판매 가능)", "예약 가능(avail365>0)", "리뷰 있음",
           "최근 1년 실거래", "휴면(1년 무거래)", "source=0 (정밀 분석용)"])
seg["비율(%)"] = (seg["건수"] / len(d) * 100).round(1)
# source=1은 과거값 이월이라 시점 의존 지표를 판정할 수 없다 → source=0 기준을 함께 본다
_s0 = d[d["source"] == 0]
seg["source=0_비율(%)"] = [
    100.0, (_s0["has_price"].mean() * 100), (_s0["is_bookable"].mean() * 100),
    (_s0["has_review"].mean() * 100), (_s0["is_active"].mean() * 100),
    (_s0["is_dormant"].mean() * 100), 100.0]
seg["source=0_비율(%)"] = seg["source=0_비율(%)"].round(1)
print(seg.to_string())
print()
print("   [주의] '휴면'은 전체 기준 37.9%지만, 이월분(source=1)을 뺀 유효 수치는 "
      f"{_s0['is_dormant'].sum():,}건 ({_s0['is_dormant'].mean():.1%})")

# 핵심 분석용 서브셋: 가격·가동률 비교가 유효한 모집단
core = d[(d["source"] == 0) & d["has_price"]].copy()
print(f"\n핵심 분석 모집단 (source=0 & 가격 보유): {len(core):,}건 ({len(core)/len(d):.1%})")

d.to_parquet("insightstay_clean.parquet", index=False)
core.to_parquet("insightstay_core.parquet", index=False)
print("\n저장 완료: insightstay_clean.parquet / insightstay_core.parquet")

In [ ]:
# [13-2] 정제 전후 비교 — 처리로 분포가 왜곡되지 않았는지 확인
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].hist(np.log10(df["price"].dropna()), bins=60, alpha=0.6, color=C["gray"], label="정제 전", density=True)
axes[0].hist(np.log10(d["price"].dropna()),  bins=60, alpha=0.6, color=C["main"], label="정제 후", density=True)
axes[0].set_xlabel("log10(price)"); axes[0].legend()
axes[0].set_title("price 분포 비교", loc="left", fontweight="bold")

axes[1].hist(df["minimum_nights"].dropna().clip(upper=40), bins=40, alpha=0.6, color=C["gray"], density=True)
axes[1].hist(d["minimum_nights"].dropna().clip(upper=40),  bins=40, alpha=0.6, color=C["main"], density=True)
axes[1].set_xlabel("minimum_nights"); axes[1].set_title("minimum_nights 분포 비교", loc="left", fontweight="bold")

axes[2].hist(df["availability_365"], bins=40, alpha=0.6, color=C["gray"], density=True)
axes[2].hist(d["availability_365"],  bins=40, alpha=0.6, color=C["main"], density=True)
axes[2].set_xlabel("availability_365"); axes[2].set_title("availability_365 분포 비교", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

print("정제 후 잔여 결측 (상위 12)")
print((d.isna().mean() * 100).sort_values(ascending=False).head(12).round(2).to_frame("pct_missing").to_string())

## 14. EDA 요약 — 다음 분석으로 넘길 5가지

1. **`source`는 데이터 갱신 여부다.** `1`(19.4%)은 갱신 실패로 **과거값이 이월된 행**이라 가격·욕실·침대가 없고
   달력은 0으로 채워져 있다. 같은 호스트로 비교해도 `avail=0` 비율이 0.1% vs 50.4%. 도시별 비중이 0~48%로 달라,
   통제 없이 비교하면 **갱신 실패율 차이를 시장 차이로 오독**한다.
   → 시점이 개입하는 분석은 `insightstay_core.parquet`(source=0 & 가격 보유, 약 65만 건) 기준.

2. **`price` = 수수료 포함 1박 환산가, `list_price` = 정가.** 총액(`price_quote_total_price`)은 견적 기간이
   1~100박으로 제각각이라 객단가로 쓰면 안 된다. 할인은 13.8%에만 적용되고 `discount_type` 과 100% 대응.

3. **죽은 재고가 크다 — 단 근거는 하나만.** `number_of_reviews_ltm == 0` 기준 **source=0에서 30.5%(207,005건)**.
   `availability_365 == 0`(13.3%)은 **99.5%가 source=1의 미수집**이라 지표로 쓸 수 없다.

4. **`minimum_nights` 는 전환에 직접 영향을 준다.** 가동률 1박 44% → 4-7박 55% → **30박+ 32%로 급락**,
   객단가도 205€ → 88€로 반토막. 30박+ 세그먼트(9.0%)가 수익성 최악 구간.

5. **도시 규모 ≠ 수익성.** 런던(공급 1위)·시칠리아(3위)의 추정 연매출 중앙값은 2.2만/1.2만€인데,
   마요르카·메노르카는 6.7~6.9만€. **공급이 몰린 곳과 돈이 되는 곳이 다르다.**

> **결측 처리 최종 결론** — 삭제는 **0.12%(1,039건)** 의 MCAR 소량 결측과 **0.48%** 의 명백한 오류 행에만 적용.
> 나머지 대규모 결측(price 18.6%, 리뷰 19.7%, bathrooms 25.1%)은 **전부 구조적 결측이라 대체하지 않고 플래그로 분리**했다.
> 채우지 않은 것이 게으름이 아니라, **"비어 있다"가 이 과제에서 가장 중요한 신호이기 때문**이다.